# Conflict-Induced Food Crisis Prediction
## Task 1: Data Fusion & Panel Creation  v2.0

**Goal**: Fuse ACLED conflict, FEWS NET IPC, and CHIRPS rainfall into a single
`country × admin1 × year_month` panel ready for feature engineering.

| Step | Action | Output |
|------|--------|--------|
| 1 | Install & import | environment ready |
| 2 | Configuration | country lists, date range |
| 3 | Load ACLED | `acled_africa_raw.csv` → monthly aggregation |
| 4 | Load CHIRPS | `chirps_monthly.csv` → anomaly features |
| 5 | Load FEWS NET | IPC phases → `fewsnet_ipc.csv` |
| 6 | Three-way merge | `panel_dataset.csv` (14,697 rows) |
| 7 | Quality report | `data_quality_report.json` |
| 8 | Drive backup | all artifacts saved |

**Design rules**
- FEWS NET is the LEFT table → only labeled region-months survive the merge  
- `crisis_90d` = 1 if IPC Phase ≥ 3 in ANY of the next 3 months (forward-only)  
- No future leakage: features use current/past data only  
- Uganda excluded: ACLED at admin-2, FEWS at admin-1 → 99 % unmatched  


---
## Step 1 — Install & Import

In [12]:
# ── Install (Colab only — safe to re-run) ───────────────────────────────────
!pip install requests geopandas shapely rapidfuzz -q

import os, json, time, re, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
from pathlib import Path
from rapidfuzz import process, fuzz

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.3f}'.format)

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR   = Path('/content/crisis_outputs')   # raw inputs from Data Collection
OUTPUT_DIR = DATA_DIR                           # same directory for simplicity
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Data / output dir : {DATA_DIR}')
print(f'Pandas version    : {pd.__version__}')
print(' All libraries loaded.')


Data / output dir : /content/crisis_outputs
Pandas version    : 2.2.2
 All libraries loaded.


---
## Step 2 — Google Drive Mount

In [13]:
# ── Mount Drive & restore backup ────────────────────────────────────────────
from google.colab import drive
import shutil

drive.mount('/content/drive')

BACKUP_SRC = Path('/content/drive/MyDrive/crisis_outputs_backup')

if BACKUP_SRC.exists():
    if not any(DATA_DIR.iterdir()):
        print(f'Restoring from {BACKUP_SRC} ...')
        for item in BACKUP_SRC.iterdir():
            dst = DATA_DIR / item.name
            shutil.copytree(str(item), str(dst), dirs_exist_ok=True) if item.is_dir()                 else shutil.copy2(str(item), str(dst))
        print(' Restore complete.')
    else:
        print(' Files already present — skipping restore.')
else:
    print('  No backup found — will run all steps from scratch.')

# List what we have
existing = sorted(DATA_DIR.glob('*.csv'))
print(f'\nExisting CSVs ({len(existing)}):')
for f in existing:
    print(f'  {f.name:<45} {f.stat().st_size/1024:>8.1f} KB')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Files already present — skipping restore.

Existing CSVs (26):
  X_test.csv                                      1289.3 KB
  X_train.csv                                     2207.3 KB
  X_val.csv                                        621.3 KB
  acled_africa_raw.csv                           10334.1 KB
  acled_monthly.csv                               1766.5 KB
  chirps_monthly.csv                              2016.7 KB
  chirps_processed.csv                            2016.7 KB
  feature_importance.csv                             2.0 KB
  features_engineered.csv                         5791.2 KB
  features_manifest.csv                              1.4 KB
  fews_net_ipc_admin1 (1).csv                   311841.7 KB
  fews_net_ipc_admin1.csv                       311504.7 KB
  fewsnet_ipc.csv                                 1027.6 KB
  model_dataset.csv        

---
## Step 3 — Configuration

In [14]:
# ── Global settings ──────────────────────────────────────────────────────────
START_DATE = '2018-01-01'
END_DATE   = '2026-01-01'

# All 20 ACLED countries
ALL_COUNTRIES = [
    'Ethiopia','Somalia','Sudan','South Sudan','Kenya',
    'Uganda','Nigeria','Niger','Mali','Burkina Faso',
    'Chad','Cameroon','Mozambique','Zimbabwe','Malawi',
    'Democratic Republic of Congo','Central African Republic',
    'Madagascar','Eritrea','Burundi',
]

# These 5 countries are ABSENT from the FEWS NET release used
FEWS_ABSENT   = {'Burundi','Eritrea','Madagascar','Malawi','Zimbabwe'}

# Uganda: ACLED at admin-2, FEWS at admin-1 → not compatible
PANEL_EXCLUDE = FEWS_ABSENT | {'Uganda'}

PANEL_COUNTRIES = [c for c in ALL_COUNTRIES if c not in PANEL_EXCLUDE]

# ACLED OAuth credentials — move to Colab Secrets in production
try:
    from google.colab import userdata
    ACLED_EMAIL    = userdata.get('ACLED_EMAIL')
    ACLED_PASSWORD = userdata.get('ACLED_PASSWORD')
    print(' Credentials loaded from Colab Secrets.')
except Exception:
    ACLED_EMAIL    = 'dule.abera_mif24@nusaputra.ac.id'
    ACLED_PASSWORD = '$$Dul411829'
    print('  Using hardcoded credentials (set Colab Secrets for safety).')

print(f'\nDate range      : {START_DATE} → {END_DATE}')
print(f'All countries   : {len(ALL_COUNTRIES)}')
print(f'Excluded        : {sorted(PANEL_EXCLUDE)}')
print(f'Panel countries : {len(PANEL_COUNTRIES)} → {PANEL_COUNTRIES}')


  Using hardcoded credentials (set Colab Secrets for safety).

Date range      : 2018-01-01 → 2026-01-01
All countries   : 20
Excluded        : ['Burundi', 'Eritrea', 'Madagascar', 'Malawi', 'Uganda', 'Zimbabwe']
Panel countries : 14 → ['Ethiopia', 'Somalia', 'Sudan', 'South Sudan', 'Kenya', 'Nigeria', 'Niger', 'Mali', 'Burkina Faso', 'Chad', 'Cameroon', 'Mozambique', 'Democratic Republic of Congo', 'Central African Republic']


---
## Step 4 — Load & Aggregate ACLED Conflict Data

In [15]:
# ── ACLED: load existing or download fresh ───────────────────────────────────
RAW_ACLED = DATA_DIR / 'acled_africa_raw.csv'

def get_acled_token(email, password):
    r = requests.post(
        'https://acleddata.com/oauth/token',
        data={'username': email, 'password': password,
              'grant_type': 'password', 'client_id': 'acled'},
        timeout=20
    )
    r.raise_for_status()
    print(' ACLED token acquired.')
    return r.json()['access_token']

if RAW_ACLED.exists():
    print(f'Loading existing ACLED file: {RAW_ACLED.name}')
    acled_raw = pd.read_csv(RAW_ACLED)
    acled_raw['event_date'] = pd.to_datetime(acled_raw['event_date'])
else:
    print('Downloading ACLED data (paginated, no cap)...')
    token   = get_acled_token(ACLED_EMAIL, ACLED_PASSWORD)
    headers = {'Authorization': f'Bearer {token}'}
    all_events = []
    for country in ALL_COUNTRIES:
        page, country_events = 1, []
        while True:
            params = {
                'country': country,
                'event_date': f'{START_DATE}|{END_DATE}',
                'event_date_where': 'BETWEEN',
                'fields': 'event_date|event_type|country|admin1|fatalities',
                'limit': 5000, 'page': page,
            }
            try:
                r = requests.get('https://acleddata.com/api/acled/read',
                                 headers=headers, params=params, timeout=60)
                data = r.json().get('data', [])
                if not data: break
                country_events.extend(data)
                if len(data) < 5000: break
                page += 1; time.sleep(1)
            except Exception as e:
                print(f'   {country} p{page}: {e}'); break
        print(f'  {country:<30} {len(country_events):>7,} events')
        all_events.extend(country_events)
    acled_raw = pd.DataFrame(all_events)
    acled_raw['event_date'] = pd.to_datetime(acled_raw['event_date'])
    acled_raw['fatalities'] = pd.to_numeric(acled_raw['fatalities'], errors='coerce').fillna(0)
    acled_raw.to_csv(RAW_ACLED, index=False)

acled_raw['fatalities'] = pd.to_numeric(acled_raw['fatalities'], errors='coerce').fillna(0)
print(f'\nTotal events  : {len(acled_raw):,}')
print(f'Countries     : {acled_raw["country"].nunique()}')
print(f'Date range    : {acled_raw["event_date"].min().date()} → {acled_raw["event_date"].max().date()}')
print(f'Fatalities    : {acled_raw["fatalities"].sum():,.0f}')
print(f'\nEvents per country:')
print(acled_raw['country'].value_counts().to_string())


Loading existing ACLED file: acled_africa_raw.csv

Total events  : 207,682
Countries     : 20
Date range    : 2018-01-01 → 2025-04-01
Fatalities    : 355,826

Events per country:
country
Nigeria                         29998
Democratic Republic of Congo    29319
Somalia                         24573
Sudan                           24193
Cameroon                        16404
Burkina Faso                    11406
Kenya                           10984
Ethiopia                        10890
Mali                            10864
South Sudan                     10311
Uganda                           4436
Niger                            4347
Burundi                          4296
Central African Republic         4213
Mozambique                       4167
Madagascar                       3166
Zimbabwe                         1434
Chad                             1378
Malawi                           1245
Eritrea                            58


---
## Step 5 — ACLED Monthly Aggregation

In [16]:
# ── Monthly aggregation per country × admin1 ─────────────────────────────────
acled_raw['year_month']    = acled_raw['event_date'].dt.to_period('M')
acled_raw['country_clean'] = acled_raw['country'].str.strip()
acled_raw['admin1_clean']  = acled_raw['admin1'].str.strip().str.title()

BATTLE_TYPES   = {'Battles', 'Explosions/Remote violence'}
CIVILIAN_TYPES = {'Violence against civilians'}

monthly = (
    acled_raw
    .groupby(['country_clean', 'admin1_clean', 'year_month'])
    .apply(lambda g: pd.Series({
        'fatalities_30d'   : g['fatalities'].sum(),
        'events_30d'       : len(g),
        'battle_events'    : g['event_type'].isin(BATTLE_TYPES).sum(),
        'civilian_violence': g['event_type'].isin(CIVILIAN_TYPES).sum(),
    }))
    .reset_index()
    .rename(columns={'country_clean': 'country', 'admin1_clean': 'admin1'})
)

# ── Zero-fill missing months per region ──────────────────────────────────────
all_months = pd.period_range(monthly['year_month'].min(),
                              monthly['year_month'].max(), freq='M')
full_parts = []
for (country, admin1), grp in monthly.groupby(['country', 'admin1']):
    idx = pd.MultiIndex.from_product(
        [[country], [admin1], all_months],
        names=['country', 'admin1', 'year_month']
    )
    full_parts.append(grp.set_index(['country', 'admin1', 'year_month']).reindex(idx))

monthly = pd.concat(full_parts).reset_index()
fill_cols = ['fatalities_30d', 'events_30d', 'battle_events', 'civilian_violence']
monthly[fill_cols] = monthly[fill_cols].fillna(0)

# ── Conflict trend: +1 escalating / 0 stable / -1 de-escalating ──────────────
monthly = monthly.sort_values(['country', 'admin1', 'year_month'])
prev_avg = monthly.groupby(['country', 'admin1'])['fatalities_30d'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)
threshold = 5
monthly['conflict_trend'] = 0
monthly.loc[(monthly['fatalities_30d'] > prev_avg + threshold) |
            ((prev_avg == 0) & (monthly['fatalities_30d'] > 0)), 'conflict_trend'] = 1
monthly.loc[(monthly['conflict_trend'] == 0) &
            ((monthly['fatalities_30d'] < prev_avg - threshold) |
             ((monthly['fatalities_30d'] == 0) & (prev_avg > 0))), 'conflict_trend'] = -1
monthly['conflict_trend'] = monthly['conflict_trend'].fillna(0).astype(int)
monthly['year_month'] = monthly['year_month'].astype(str)

monthly.to_csv(DATA_DIR / 'acled_monthly.csv', index=False)

print(' ACLED monthly aggregated:')
print(f'  Rows      : {len(monthly):,}')
print(f'  Countries : {monthly["country"].nunique()}')
print(f'  Regions   : {monthly["admin1"].nunique()}')
print(f'  Columns   : {monthly.columns.tolist()}')
monthly.head(3)


 ACLED monthly aggregated:
  Rows      : 40,128
  Countries : 20
  Regions   : 450
  Columns   : ['country', 'admin1', 'year_month', 'fatalities_30d', 'events_30d', 'battle_events', 'civilian_violence', 'conflict_trend']


,country,admin1,year_month,fatalities_30d,events_30d,battle_events,civilian_violence,conflict_trend
0,Burkina Faso,Boucle Du Mouhoun,2018-01,0.000,1.000,1.000,0.000,0
1,Burkina Faso,Boucle Du Mouhoun,2018-02,0.000,0.000,0.000,0.000,0
2,Burkina Faso,Boucle Du Mouhoun,2018-03,1.000,1.000,1.000,0.000,1


---
## Step 6 — Process CHIRPS Rainfall (Zonal Stats → Anomalies)

In [17]:
# ── Load the admin-1 zonal CHIRPS from Data Collection Module ────────────────
chirps_path = DATA_DIR / 'chirps_monthly.csv'
chirps_raw  = pd.read_csv(chirps_path)

print(f'Loaded CHIRPS: {len(chirps_raw):,} rows')
print(f'Columns      : {chirps_raw.columns.tolist()}')
print(f'Sample head  :\n{chirps_raw.head(3).to_string()}')

# ── Normalise column names ───────────────────────────────────────────────────
chirps_raw = chirps_raw.rename(columns={'average_rainfall_mm': 'rainfall_mm'})

# Build year_month if absent
if 'year_month' not in chirps_raw.columns and 'year' in chirps_raw.columns:
    chirps_raw['year_month'] = (chirps_raw['year'].astype(str) + '-' +
                                chirps_raw['month'].astype(str).str.zfill(2))

# Normalise key columns to lowercase for consistent merging
chirps_raw['country']    = chirps_raw['country'].astype(str).str.strip().str.lower()
chirps_raw['admin1']     = chirps_raw['admin'].astype(str).str.strip().str.lower()                            if 'admin' in chirps_raw.columns                            else chirps_raw['admin1'].astype(str).str.strip().str.lower()
chirps_raw['year_month'] = chirps_raw['year_month'].astype(str)

# ── Deduplicate ──────────────────────────────────────────────────────────────
before = len(chirps_raw)
chirps_raw = chirps_raw.drop_duplicates(subset=['country', 'admin1', 'year_month'])
print(f'\nAfter dedup: {len(chirps_raw):,} rows (removed {before - len(chirps_raw):,})')

# ── Seasonal anomaly per (country, admin1, calendar-month) ───────────────────
chirps_raw = chirps_raw.drop(columns=['rainfall_anomaly','is_drought','is_flood'], errors='ignore')
chirps_raw['month_num'] = pd.to_numeric(chirps_raw['year_month'].str[-2:], errors='coerce')
chirps_raw = chirps_raw.dropna(subset=['month_num'])

seas = (chirps_raw
        .groupby(['country','admin1','month_num'])['rainfall_mm']
        .agg(['mean','std']).reset_index()
        .rename(columns={'mean':'seas_mean','std':'seas_std'}))
chirps_raw = chirps_raw.merge(seas, on=['country','admin1','month_num'], how='left')

chirps_raw['rainfall_anomaly'] = (
    (chirps_raw['rainfall_mm'] - chirps_raw['seas_mean'])
    / chirps_raw['seas_std'].replace(0, np.nan)
).round(3)
chirps_raw['is_drought'] = (chirps_raw['rainfall_anomaly'] < -0.5).astype(int)
chirps_raw['is_flood']   = (chirps_raw['rainfall_anomaly'] >  1.5).astype(int)

chirps_df = chirps_raw[['country','admin1','year_month',
                         'rainfall_mm','rainfall_anomaly','is_drought','is_flood']]             .fillna({'rainfall_anomaly': 0, 'is_drought': 0, 'is_flood': 0})
chirps_df.to_csv(DATA_DIR / 'chirps_processed.csv', index=False)

print(f'\n CHIRPS processed:')
print(f'  Rows          : {len(chirps_df):,}')
print(f'  Countries     : {chirps_df["country"].nunique()}')
print(f'  Admin regions : {chirps_df["admin1"].nunique()}')
print(f'  Drought ratio : {chirps_df["is_drought"].mean():.1%}')
print(f'  Flood ratio   : {chirps_df["is_flood"].mean():.1%}')
chirps_df.head(3)


Loaded CHIRPS: 44,523 rows
Columns      : ['country', 'admin1', 'year_month', 'rainfall_mm', 'rainfall_anomaly', 'is_drought', 'is_flood']
Sample head  :
        country             admin1 year_month  rainfall_mm  rainfall_anomaly  is_drought  is_flood
0  burkina faso  boucle du mouhoun    2018-01       44.612            -1.406           1         0
1  burkina faso  boucle du mouhoun    2018-02       58.063             1.990           0         1
2  burkina faso  boucle du mouhoun    2018-03       63.212             1.041           0         0

After dedup: 44,523 rows (removed 0)

 CHIRPS processed:
  Rows          : 44,523
  Countries     : 20
  Admin regions : 451
  Drought ratio : 28.9%
  Flood ratio   : 7.2%


,country,admin1,year_month,rainfall_mm,rainfall_anomaly,is_drought,is_flood
0,burkina faso,boucle du mouhoun,2018-01,44.612,-1.406,1,0
1,burkina faso,boucle du mouhoun,2018-02,58.063,1.990,0,1
2,burkina faso,boucle du mouhoun,2018-03,63.212,1.041,0,0


---
## Step 7 — Load & Process FEWS NET IPC Data

In [18]:
# ── Load the admin-1 zonal CHIRPS from Data Collection Module ────────────────
chirps_path = DATA_DIR / 'chirps_monthly.csv'
chirps_raw  = pd.read_csv(chirps_path)

print(f'Loaded CHIRPS: {len(chirps_raw):,} rows')
print(f'Columns      : {chirps_raw.columns.tolist()}')
print(f'Sample head  :\n{chirps_raw.head(3).to_string()}')

# ── Normalise column names ───────────────────────────────────────────────────
chirps_raw = chirps_raw.rename(columns={'average_rainfall_mm': 'rainfall_mm'})

# Build year_month if absent
if 'year_month' not in chirps_raw.columns and 'year' in chirps_raw.columns:
    chirps_raw['year_month'] = (chirps_raw['year'].astype(str) + '-' +
                                chirps_raw['month'].astype(str).str.zfill(2))

# Normalise key columns to lowercase for consistent merging
chirps_raw['country']    = chirps_raw['country'].astype(str).str.strip().str.lower()
chirps_raw['admin1']     = chirps_raw['admin'].astype(str).str.strip().str.lower()                            if 'admin' in chirps_raw.columns                            else chirps_raw['admin1'].astype(str).str.strip().str.lower()
chirps_raw['year_month'] = chirps_raw['year_month'].astype(str)

# ── Deduplicate ──────────────────────────────────────────────────────────────
before = len(chirps_raw)
chirps_raw = chirps_raw.drop_duplicates(subset=['country', 'admin1', 'year_month'])
print(f'\nAfter dedup: {len(chirps_raw):,} rows (removed {before - len(chirps_raw):,})')

# ── Seasonal anomaly per (country, admin1, calendar-month) ───────────────────
chirps_raw = chirps_raw.drop(columns=['rainfall_anomaly','is_drought','is_flood'], errors='ignore')
chirps_raw['month_num'] = pd.to_numeric(chirps_raw['year_month'].str[-2:], errors='coerce')
chirps_raw = chirps_raw.dropna(subset=['month_num'])

seas = (chirps_raw
        .groupby(['country','admin1','month_num'])['rainfall_mm']
        .agg(['mean','std']).reset_index()
        .rename(columns={'mean':'seas_mean','std':'seas_std'}))
chirps_raw = chirps_raw.merge(seas, on=['country','admin1','month_num'], how='left')

chirps_raw['rainfall_anomaly'] = (
    (chirps_raw['rainfall_mm'] - chirps_raw['seas_mean'])
    / chirps_raw['seas_std'].replace(0, np.nan)
).round(3)
chirps_raw['is_drought'] = (chirps_raw['rainfall_anomaly'] < -0.5).astype(int)
chirps_raw['is_flood']   = (chirps_raw['rainfall_anomaly'] >  1.5).astype(int)

chirps_df = chirps_raw[['country','admin1','year_month',
                         'rainfall_mm','rainfall_anomaly','is_drought','is_flood']]             .fillna({'rainfall_anomaly': 0, 'is_drought': 0, 'is_flood': 0})
chirps_df.to_csv(DATA_DIR / 'chirps_processed.csv', index=False)

print(f'\n CHIRPS processed:')
print(f'  Rows          : {len(chirps_df):,}')
print(f'  Countries     : {chirps_df["country"].nunique()}')
print(f'  Admin regions : {chirps_df["admin1"].nunique()}')
print(f'  Drought ratio : {chirps_df["is_drought"].mean():.1%}')
print(f'  Flood ratio   : {chirps_df["is_flood"].mean():.1%}')
chirps_df.head(3)


Loaded CHIRPS: 44,523 rows
Columns      : ['country', 'admin1', 'year_month', 'rainfall_mm', 'rainfall_anomaly', 'is_drought', 'is_flood']
Sample head  :
        country             admin1 year_month  rainfall_mm  rainfall_anomaly  is_drought  is_flood
0  burkina faso  boucle du mouhoun    2018-01       44.612            -1.406           1         0
1  burkina faso  boucle du mouhoun    2018-02       58.063             1.990           0         1
2  burkina faso  boucle du mouhoun    2018-03       63.212             1.041           0         0

After dedup: 44,523 rows (removed 0)

 CHIRPS processed:
  Rows          : 44,523
  Countries     : 20
  Admin regions : 451
  Drought ratio : 28.9%
  Flood ratio   : 7.2%


,country,admin1,year_month,rainfall_mm,rainfall_anomaly,is_drought,is_flood
0,burkina faso,boucle du mouhoun,2018-01,44.612,-1.406,1,0
1,burkina faso,boucle du mouhoun,2018-02,58.063,1.990,0,1
2,burkina faso,boucle du mouhoun,2018-03,63.212,1.041,0,0


---
## Step 8 — Three-Way Panel Merge

In [19]:
# ── Admin-1 name normaliser ───────────────────────────────────────────────────
def std_name(name):
    if pd.isna(name): return ''
    name = str(name).strip().lower()
    name = re.sub(r'[^\w\s]', ' ', name)
    name = re.sub(r'\s+', ' ', name).strip()
    fixes = {
        'southern nations nationalities and peoples': 'snnpr',
        'centre nord':'centre-nord','centre est':'centre-est',
        'centre ouest':'centre-ouest','hauts bassins':'hauts-bassins',
        'extreme nord':'extreme-nord','far north':'extreme-nord',
    }
    for old, new in fixes.items():
        if old in name: return new
    return name

# ── Reload processed files ───────────────────────────────────────────────────
acled_m  = pd.read_csv(DATA_DIR / 'acled_monthly.csv')
fews_m   = pd.read_csv(DATA_DIR / 'fewsnet_ipc.csv')
chirps_m = pd.read_csv(DATA_DIR / 'chirps_processed.csv')

print(f'ACLED  : {len(acled_m):,} rows')
print(f'FEWS   : {len(fews_m):,} rows')
print(f'CHIRPS : {len(chirps_m):,} rows')

# ── Build normalised keys ─────────────────────────────────────────────────────
for df in [acled_m, fews_m, chirps_m]:
    df['country_key'] = df['country'].astype(str).str.strip().str.lower()
    df['admin1_key']  = df['admin1'].apply(std_name)
    df['year_month']  = df['year_month'].astype(str)

# ── Fuzzy admin1 mapping: ACLED/CHIRPS → FEWS name space ─────────────────────
fews_dict = fews_m.groupby('country_key')['admin1_key'].unique().to_dict()
admin1_map = {}
for country in acled_m['country_key'].unique():
    for ac in acled_m[acled_m['country_key']==country]['admin1_key'].unique():
        fews_names = fews_dict.get(country, [])
        if len(fews_names) == 0: continue
        match = process.extractOne(ac, fews_names, scorer=fuzz.token_sort_ratio, score_cutoff=70)
        admin1_map[(country, ac)] = match[0] if match else ac
print(f'\nFuzzy map size: {len(admin1_map)} ACLED→FEWS region pairs')

for df in [acled_m, chirps_m]:
    df['admin1_merge'] = df.apply(
        lambda r: admin1_map.get((r['country_key'], r['admin1_key']), r['admin1_key']), axis=1)
fews_m['admin1_merge'] = fews_m['admin1_key']

# ── Deduplicate all three before merge ───────────────────────────────────────
key = ['country_key','admin1_merge','year_month']
acled_d  = acled_m.drop_duplicates(subset=key)
chirps_d = chirps_m.drop_duplicates(subset=key)

# ── Merge: FEWS (left) + ACLED + CHIRPS ─────────────────────────────────────
ACLED_COLS  = ['country_key','admin1_merge','year_month',
               'fatalities_30d','events_30d','battle_events','civilian_violence','conflict_trend']
CHIRPS_COLS = ['country_key','admin1_merge','year_month',
               'rainfall_mm','rainfall_anomaly','is_drought','is_flood']

panel = (fews_m
         .merge(acled_d[ACLED_COLS],  on=key, how='left')
         .merge(chirps_d[CHIRPS_COLS], on=key, how='left'))

# Zero-fill missing conflict (no events = safe month)
conflict_cols = ['fatalities_30d','events_30d','battle_events','civilian_violence','conflict_trend']
panel[conflict_cols] = panel[conflict_cols].fillna(0)

# Fill missing rainfall with country median per month
panel['rainfall_mm'] = panel.groupby(['country_key','year_month'])['rainfall_mm']                             .transform(lambda x: x.fillna(x.median()))
panel['rainfall_anomaly'] = panel['rainfall_anomaly'].fillna(0)
panel[['is_drought','is_flood']] = panel[['is_drought','is_flood']].fillna(0)

# Lean / Harvest season flags
panel['month_num'] = pd.to_numeric(panel['year_month'].str[5:7], errors='coerce')
panel['is_lean_season']    = panel['month_num'].between(6, 9).astype(int)
panel['is_harvest_season'] = panel['month_num'].between(10,12).astype(int)
panel['year'] = panel['year_month'].str[:4].astype(int)
panel['population_in_crisis'] = (panel['ipc_phase'] >= 3).astype(int)

print(f'\n Full panel: {len(panel):,} rows, {panel["country"].nunique()} countries')
print(f'  Missing conflict : {panel["fatalities_30d"].isnull().sum()}')
print(f'  Missing CHIRPS   : {panel["rainfall_mm"].isnull().sum()}')


ACLED  : 40,128 rows
FEWS   : 28,170 rows
CHIRPS : 44,523 rows

Fuzzy map size: 380 ACLED→FEWS region pairs

 Full panel: 28,170 rows, 14 countries
  Missing conflict : 0
  Missing CHIRPS   : 1304


---
## Step 9 — Production Panel (Reliable Countries Only)

In [20]:
# ── Drop excluded countries and clean up merge keys ──────────────────────────
PANEL_EXCLUDE_LOWER = {c.lower() for c in PANEL_EXCLUDE}
panel_prod = panel[~panel['country_key'].isin(PANEL_EXCLUDE_LOWER)].copy()
panel_prod = panel_prod.drop(columns=['country_key','admin1_key','admin1_merge',
                                       'month_num'], errors='ignore')

# ── Canonical column order ────────────────────────────────────────────────────
BASE_COLS    = ['country','admin1','year_month','ipc_phase','population_in_crisis','crisis_90d']
FEATURE_COLS = ['ipc_lag1','ipc_lag2',
                'fatalities_30d','events_30d','battle_events','civilian_violence','conflict_trend',
                'rainfall_mm','rainfall_anomaly','is_drought','is_flood',
                'is_lean_season','is_harvest_season','year']

panel_prod = panel_prod[BASE_COLS + FEATURE_COLS].reset_index(drop=True)
panel_prod['conflict_trend'] = panel_prod['conflict_trend'].fillna(0).astype(int)

# ── Validation checks ─────────────────────────────────────────────────────────
dup_keys = panel_prod.duplicated(subset=['country','admin1','year_month']).sum()
assert dup_keys == 0, f' {dup_keys} duplicate region-month keys found!'
assert panel_prod['crisis_90d'].between(0,1).all(), ' crisis_90d out of range!'
assert panel_prod['ipc_phase'].between(1,5).all(), ' ipc_phase out of range!'

panel_prod.to_csv(OUTPUT_DIR / 'panel_dataset.csv', index=False)

# ── Correlation check ─────────────────────────────────────────────────────────
ipc_corr      = panel_prod[['ipc_lag1','crisis_90d']].corr().iloc[0,1]
conflict_corr = panel_prod[['conflict_trend','crisis_90d']].corr().iloc[0,1]
drought_corr  = panel_prod[['is_drought','crisis_90d']].corr().iloc[0,1]

print('=== PRODUCTION PANEL ===')
print(f'  Rows              : {len(panel_prod):,}')
print(f'  Countries         : {panel_prod["country"].nunique()}')
print(f'  Regions           : {panel_prod["admin1"].nunique()}')
print(f'  Date range        : {panel_prod["year_month"].min()} → {panel_prod["year_month"].max()}')
print(f'  Crisis rate       : {panel_prod["crisis_90d"].mean():.1%}')
print(f'  Class imbalance   : {(1-panel_prod["crisis_90d"].mean())/panel_prod["crisis_90d"].mean():.1f}:1')
print(f'  Duplicate rows    : {dup_keys} ')
print(f'\nCorrelations with crisis_90d:')
print(f'  ipc_lag1       : {ipc_corr:.4f}  (target: > 0.60)')
print(f'  conflict_trend : {conflict_corr:.4f}')
print(f'  is_drought     : {drought_corr:.4f}')
print(f'\nExcluded countries:')
for c in sorted(PANEL_EXCLUDE):
    reason = 'absent from FEWS NET' if c in FEWS_ABSENT else 'admin-2/admin-1 mismatch'
    print(f'  ✗ {c}: {reason}')
print(f'\n Saved → panel_dataset.csv')
panel_prod.describe().T


=== PRODUCTION PANEL ===
  Rows              : 26,954
  Countries         : 13
  Regions           : 341
  Date range        : 2018-02 → 2026-01
  Crisis rate       : 39.2%
  Class imbalance   : 1.5:1
  Duplicate rows    : 0 

Correlations with crisis_90d:
  ipc_lag1       : 0.8440  (target: > 0.60)
  conflict_trend : 0.0320
  is_drought     : -0.0006

Excluded countries:
  ✗ Burundi: absent from FEWS NET
  ✗ Eritrea: absent from FEWS NET
  ✗ Madagascar: absent from FEWS NET
  ✗ Malawi: absent from FEWS NET
  ✗ Uganda: admin-2/admin-1 mismatch
  ✗ Zimbabwe: absent from FEWS NET

 Saved → panel_dataset.csv


,count,mean,std,min,25%,50%,75%,max
ipc_phase,26954.000,2.251,0.987,1.000,1.000,2.000,3.000,5.000
population_in_crisis,26954.000,0.372,0.483,0.000,0.000,0.000,1.000,1.000
crisis_90d,26954.000,0.392,0.488,0.000,0.000,0.000,1.000,1.000
ipc_lag1,26611.000,2.246,0.985,1.000,1.000,2.000,3.000,5.000
ipc_lag2,26268.000,2.242,0.983,1.000,1.000,2.000,3.000,5.000
fatalities_30d,26954.000,10.191,49.100,0.000,0.000,0.000,2.000,2408.000
events_30d,26954.000,5.268,15.459,0.000,0.000,0.000,4.000,545.000
battle_events,26954.000,2.163,10.199,0.000,0.000,0.000,1.000,440.000
civilian_violence,26954.000,1.436,4.377,0.000,0.000,0.000,1.000,125.000
conflict_trend,26954.000,-0.126,0.580,-1.000,0.000,0.000,0.000,1.000


---
## Step 10 — Quality Report & Validation

In [21]:
# ── Quality report JSON ───────────────────────────────────────────────────────
quality = {
    'task':                  'Task 1 — Data Fusion & Panel Creation',
    'panel_rows':            int(len(panel_prod)),
    'countries':             int(panel_prod['country'].nunique()),
    'regions':               int(panel_prod['admin1'].nunique()),
    'time_periods':          int(panel_prod['year_month'].nunique()),
    'date_start':            str(panel_prod['year_month'].min()),
    'date_end':              str(panel_prod['year_month'].max()),
    'crisis_rate':           float(round(panel_prod['crisis_90d'].mean(), 4)),
    'class_imbalance':       float(round(
        (1 - panel_prod['crisis_90d'].mean()) / panel_prod['crisis_90d'].mean(), 2)),
    'scale_pos_weight':      float(round(
        (1 - panel_prod['crisis_90d'].mean()) / panel_prod['crisis_90d'].mean(), 2)),
    'missing_conflict_pct':  float(round(panel_prod['fatalities_30d'].isnull().mean()*100, 2)),
    'missing_chirps_pct':    float(round(panel_prod['rainfall_mm'].isnull().mean()*100, 2)),
    'drought_pct':           float(round(panel_prod['is_drought'].mean()*100, 2)),
    'ipc_lag1_corr':         float(round(ipc_corr, 4)),
    'conflict_trend_corr':   float(round(conflict_corr, 4)),
    'drought_corr':          float(round(drought_corr, 4)),
    'feature_columns':       FEATURE_COLS,
    'target_column':         'crisis_90d',
    'excluded_countries':    sorted(list(PANEL_EXCLUDE)),
    'fews_absent':           sorted(list(FEWS_ABSENT)),
}

with open(OUTPUT_DIR / 'data_quality_report.json', 'w') as f:
    json.dump(quality, f, indent=2)

print('=== TASK 1 COMPLETE ===')
print(json.dumps(quality, indent=2, default=str))
print(f'\n data_quality_report.json saved')
print(f' Pass panel_dataset.csv → Task 2')


=== TASK 1 COMPLETE ===
{
  "task": "Task 1 \u2014 Data Fusion & Panel Creation",
  "panel_rows": 26954,
  "countries": 13,
  "regions": 341,
  "time_periods": 96,
  "date_start": "2018-02",
  "date_end": "2026-01",
  "crisis_rate": 0.3923,
  "class_imbalance": 1.55,
  "scale_pos_weight": 1.55,
  "missing_conflict_pct": 0.0,
  "missing_chirps_pct": 0.33,
  "drought_pct": 20.87,
  "ipc_lag1_corr": 0.844,
  "conflict_trend_corr": 0.032,
  "drought_corr": -0.0006,
  "feature_columns": [
    "ipc_lag1",
    "ipc_lag2",
    "fatalities_30d",
    "events_30d",
    "battle_events",
    "civilian_violence",
    "conflict_trend",
    "rainfall_mm",
    "rainfall_anomaly",
    "is_drought",
    "is_flood",
    "is_lean_season",
    "is_harvest_season",
    "year"
  ],
  "target_column": "crisis_90d",
  "excluded_countries": [
    "Burundi",
    "Eritrea",
    "Madagascar",
    "Malawi",
    "Uganda",
    "Zimbabwe"
  ],
  "fews_absent": [
    "Burundi",
    "Eritrea",
    "Madagascar",
    "Mala

---
## Step 11 — Backup to Google Drive

In [22]:
#import shutil
#
#drive_dest = Path('/content/drive/MyDrive/crisis_outputs_backup')
#drive_dest.mkdir(parents=True, exist_ok=True)
#
#try:
#    shutil.copytree(str(OUTPUT_DIR), str(drive_dest), dirs_exist_ok=True)
#    print(f' Backup complete → {drive_dest}')
#except Exception as e:
#    print(f' Backup error: {e}')
#
#print('\nFiles saved:')
#for f in sorted(OUTPUT_DIR.glob('*.csv')) + sorted(OUTPUT_DIR.glob('*.json')):
#    print(f'  {f.name:<45} {f.stat().st_size/1024:>8.1f} KB')
